## PageIndex Vectorless RAG

In [6]:
import os, json , time
from dotenv import load_dotenv

load_dotenv()

PAGE_INDEX_API_KEY= os.getenv("PAGE_INDEX")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [8]:
from pageindex import PageIndexClient
from groq import Groq

pi_client = PageIndexClient(api_key=PAGE_INDEX_API_KEY)
groq_client = Groq(api_key=GROQ_API_KEY)



In [10]:
## upload PDF

PDF_PATH = "sample.pdf"
print(f"Uploading:{PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"Uploaded")
print(f"Document ID:{doc_id}")
print("(save this id- you'll use it throughout the notebook)")

Uploading:sample.pdf
Uploaded
Document ID:pi-cmrhyulgw009z01qrrgglpk84
(save this id- you'll use it throughout the notebook)


In [11]:
print("Building Tree index")
print("(This runs once per document - the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"Status:{status}")

    if status == "completed":
        print("\n Tree index ready")
        break
    elif status == "failed":
        print("\n Processing failed.Check your pdf format")
        break
    time.sleep(5)

Building Tree index
(This runs once per document - the index is cached for reuse)
Status:completed

 Tree index ready


In [13]:
## Inspect The Tree Structure

tree_result = pi_client.get_tree(doc_id,node_summary=True)
pageindex_tree = tree_result.get("result",[])

print(f"Top-level sections: {len(pageindex_tree)}")
print("\n Raw tree (first node): ")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent = 2))

Top-level sections: 1

 Raw tree (first node): 
{
  "title": "FedCache: A Knowledge Cache-driven Federated Learning Architecture for Personalized Edge Intelligence",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "The paper introduces FedCache, a novel Personalized Federated Learning (PFL) architecture for Edge Intelligence designed to overcome the communication and performance limitations of existing Parameters Interaction-based and Logits Interaction-based architectures. By utilizing a server-side knowledge cache and ensemble distillation, FedCache enables efficient, personalized model training without requiring public datasets or excessive communication overhead, achieving state-of-the-art performance with significantly improved communication efficiency.",
  "text": "# FedCache: A Knowledge Cache-driven Federated Learning Architecture for Personalized Edge Intelligence\n\nZhiyuan Wu, Member, IEEE, Sheng Sun, Yuwei Wang, Member, IEEE, Min Liu, Senior Member, IEEE, Ke Xu,

In [15]:
## Preety-print
def print_tree(nodes, indent = 0):
    for node in nodes:
        prefix = " " * indent + ("-" if  indent > 0 else "")
        page = node.get("page_index","?")
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"],indent+1)

print("Full Document structure \n")
print_tree(pageindex_tree)

Full Document structure 

[0000] FedCache: A Knowledge Cache-driven Federated Learning Architecture for Personalized Edge Intelligence (p.1)
 -[0001] 1 INTRODUCTION (p.1)
 -[0002] 2 RELATED WORK (p.2)
 -[0003] 3 PRELIMINARY AND MOTIVATION (p.3)
  -[0004] 3.1 Background and Notations (p.3)
  -[0005] 3.2 Practical Limitations in Edge Intelligence (p.3)
  -[0006] 3.3 Overview of PFL Architectures (p.3)
  -[0007] 3.4 Motivation (p.4)
 -[0008] 4 KNOWLEDGE CACHE-DRIVEN PERSONALIZED FEDERATED LEARNING (p.5)
  -[0009] 4.1 System Design (p.5)
  -[0010] 4.2 Knowledge Cache (p.6)
  -[0011] 4.3 Knowledge Cache-driven Personalized Distillation (p.6)
  -[0012] 4.4 Formal Description of FedCache (p.7)
 -[0013] 5 EXPERIMENTS (p.8)
  -[0014] 5.1 Experimental Setup (p.8)
  -[0015] 5.2 Results (p.9)
   -[0016] 5.2.1 Performance on Homogeneous Models (p.9)
   -[0017] 5.2.2 Performance on Heterogeneous Models (p.9)
 -[0018] 6 ABLATION STUDY (p.11)
 -[0019] 7 DISCUSSION (p.12)
 -[0020] 8 CONCLUSION (p.13)
 

In [16]:
## Count total nodes

def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"Total nodes in tree: {total}")
print("Each node = one retrievable section of the documents")

Total nodes in tree: 23
Each node = one retrievable section of the documents


In [32]:
## LLM Tree search
def llm_tree_search(query: str, tree: list, model: str = "llama-3.3-70b-versatile") -> dict:
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title": n["title"],
                "page": n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out                      # <-- fixed: dedented out of the loop

    compressed_tree = compress(tree)    # <-- fixed: was compress(true)

    prompt = f""" you are given a query and a document's tree structure (like a Table of contents).
    Your task identify which node IDs most likely contain the answer to the query.
    Think step-by-step about which sections are relevant.
    
    Query:{query}
    Document Tree:
    {json.dumps(compressed_tree, indent=2)}
    Reply ONLY in this exact JSON format:
    {{
        "thinking":"<your step-by-step reasoning>",
        "node_list":["node_id1","node_id2"]
    }} """

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )

    raw_output = response.choices[0].message.content.strip()   # <-- fixed: added this line

    if raw_output.startswith("```"):
        raw_output = raw_output.split("```")[1]
        if raw_output.startswith("json"):
            raw_output = raw_output[4:]
        raw_output = raw_output.strip()

    try:
        result = json.loads(raw_output)
    except json.JSONDecodeError:
        result = {"thinking": raw_output, "relevant_node_ids": []}

    return result

In [33]:
query = "What is the fedcache in the research paper"

print(f"Query:{query}")
result = llm_tree_search(query, pageindex_tree)

print("LLM Reasoning : ")
print(result.get("thinking","N/A"))
print()
print("Selected node IDs",result.get("node_list",[]))

Query:What is the fedcache in the research paper
LLM Reasoning : 
To find the node IDs that most likely contain the answer to the query 'What is the fedcache in the research paper', we need to analyze the document tree structure. The query is asking about 'fedcache', which is a specific concept in the paper. We can start by looking for nodes that have 'fedcache' in their title or summary. The root node with node_id '0000' has 'FedCache' in its title, indicating that the paper is about FedCache. However, the question is asking for a description or explanation of FedCache, which is likely to be found in the introduction, methodology, or system design sections. Node '0001' is the introduction, which might provide an overview of FedCache. Node '0008' and its children ('0009', '0010', '0011', '0012') seem particularly relevant as they discuss 'KNOWLEDGE CACHE-DRIVEN PERSONALIZED FEDERATED LEARNING' and have 'FedCache' mentioned in their summaries. These nodes are likely to contain detailed 

In [49]:
## Full END to END RAG Pipeline
def find_nodes_by_ids(tree:list,target_ids: list ) -> list:
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
          found.append(node)
        if node.get("nodes"):
           found.extend(find_nodes_by_ids(node["nodes"],target_ids))
    return found 


In [53]:
def generate_answer(query:str, nodes:list, model:str="llama-3.3-70b-versatile") -> str:
    if not nodes:
        return "No relevant sections found in the documents"
    
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section : '{node['title']}' | Page{node.get('page_index','?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)

    prompt = f""" You are an expert document analyst.
    Answer the question using ONLY the provided context.
    For every claim you make, cite the section title and page number in parenthesis.
    Be concise and precise.
    
    Question:{query}
    
    Context:
    {context}
    Answer:"""

    response = groq_client.chat.completions.create(
        model = model,
        messages = [{"role":"user","content":prompt}]
      )
    return response.choices[0].message.content

In [54]:
def vectorless_rag(query:str, tree:list, verbose:bool =True) -> str:
    if verbose:
        print(f"{'='*55}")
        print(f"Query:{query}")
        print(f"{'='*55}")

    search_result = llm_tree_search(query,tree)
    node_ids = search_result.get("node_list",[])

    if verbose:
        print(f"Reasoning: {search_result.get('thinking',' ')[:200]}...")
        print(f"Retrieved node IDs: {node_ids}")

    nodes = find_nodes_by_ids(tree,node_ids)

    if verbose:
        print(f"Sections found: {[n['title'] for n in nodes]}")
    
    answer = generate_answer(query,nodes)

    if verbose:
        print(f"Answer:\n{answer}")

    return answer

In [55]:
answer = vectorless_rag(
    query = "What is federated learning?",
    tree = pageindex_tree
)

Query:What is federated learning?
Reasoning: To identify which node IDs most likely contain the answer to the query 'What is federated learning?', we start by examining the titles and summaries of the top-level nodes. The query suggests we are l...
Retrieved node IDs: ['0001', '0002', '0003', '0004', '0008']
Sections found: ['1 INTRODUCTION', '2 RELATED WORK', '3 PRELIMINARY AND MOTIVATION', '3.1 Background and Notations', '4 KNOWLEDGE CACHE-DRIVEN PERSONALIZED FEDERATED LEARNING']
Answer:
Federated Learning (FL) is a privacy-preserving distributed learning paradigm that enables multiple data owners to collaboratively train AI models without sharing owners' private data (1 INTRODUCTION, Page1).
